In [ ]:
from pyspark.sql.functions import col

from olist_silver.transformations import (
    is_valid_uuid,
    merge_into,
    null_invalid_uuid,
    parse_timestamp,
    with_processed_timestamp,
)

In [ ]:
catalog = dbutils.widgets.get("catalog")

bronze_schema = dbutils.widgets.get("bronze_schema")
raw_olist_orders_table_name = dbutils.widgets.get("raw_olist_orders_table")

silver_schema = dbutils.widgets.get("silver_schema")
orders_table_name = dbutils.widgets.get("orders_table")

In [ ]:
raw_olist_orders_df = spark.table(f"{catalog}.{bronze_schema}.{raw_olist_orders_table_name}")

In [ ]:
if not spark.catalog.tableExists(f"{catalog}.{silver_schema}.{orders_table_name}"):
    spark.sql(
        f"""
        CREATE TABLE {catalog}.{silver_schema}.{orders_table_name} (
            orderId STRING,
            customerId STRING,
            orderStatus STRING,
            orderPurchaseTimestamp TIMESTAMP,
            orderApprovedAt TIMESTAMP,
            orderDeliveredCarrierDate TIMESTAMP,
            orderDeliveredCustomerDate TIMESTAMP,
            orderEstimatedDeliveryDate TIMESTAMP,
            processedTimestamp TIMESTAMP
        )
        TBLPROPERTIES (
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
        """
    )

In [ ]:
orders_silver_df = with_processed_timestamp(
    raw_olist_orders_df.where(is_valid_uuid("order_id"))
    .select(
        col("order_id").cast("string").alias("orderId"),
        null_invalid_uuid("customer_id").cast("string").alias("customerId"),
        col("order_status").cast("string").alias("orderStatus"),
        parse_timestamp("order_purchase_timestamp").alias("orderPurchaseTimestamp"),
        parse_timestamp("order_approved_at").alias("orderApprovedAt"),
        parse_timestamp("order_delivered_carrier_date").alias("orderDeliveredCarrierDate"),
        parse_timestamp("order_delivered_customer_date").alias("orderDeliveredCustomerDate"),
        parse_timestamp("order_estimated_delivery_date").alias("orderEstimatedDeliveryDate"),
    )
    .dropDuplicates(["orderId"])
)

In [ ]:
merge_into(
    spark,
    target=f"{catalog}.{silver_schema}.{orders_table_name}",
    source_view="orders_silver_view",
    keys=["orderId"],
    source_df=orders_silver_df,
)